<a href="https://colab.research.google.com/github/jm5155/PYTHON/blob/main/Padilla_SetA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Step 1: Data Acquisition from JSON**

In [15]:
import pandas as pd
import numpy as np


json_data = """
[
  {"trans_id": "T-101", "category": "Electronics", "price": 500.0, "qty": 1},
  {"trans_id": "T-102", "category": "ELEC", "price": 450.0, "qty": 2},
  {"trans_id": "T-103", "category": "Electronics", "price": -50.0, "qty": 1},
  {"trans_id": "T-104", "category": "Home & Kitchen", "price": 25.0, "qty": 10},
  {"trans_id": "T-101", "category": "Electronics", "price": 500.0, "qty": 1},
  {"trans_id": "T-105", "category": "home", "price": null, "qty": 1}
]
"""
df = pd.read_json(json_data)


np.random.seed(42)
extra_rows = 100
extra_data = {
    "trans_id": [f"T-{106+i}" for i in range(extra_rows)],
    "category": np.random.choice(["Electronics", "Home & Kitchen", "Books"], extra_rows),
    "price": np.random.normal(100, 50, extra_rows).round(2),
    "qty": np.random.randint(1, 5, extra_rows)
}
df_extra = pd.DataFrame(extra_data)
df = pd.concat([df, df_extra], ignore_index=True)


df.loc[[7, 8], "price"] = np.nan # Missing prices
df.loc[[9, 10], "category"] = "ELECTRONICS" # Inconsistency
df.loc[50, "price"] = 1000000 # Extreme Outlier

print("Data acquired. Total records:", len(df))

Data acquired. Total records: 106


/tmp/ipykernel_2356/4292399879.py:15: FutureWarning: Passing literal json to 'read_json' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_json(json_data)


**Step 2: Data Profiling & Audit Table**

In [16]:
print("DataFrame Info:")
df.info()

print("\nDataFrame Description:")
display(df.describe())

print("\nMissing Values:")
display(df.isnull().sum())

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 106 entries, 0 to 105
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   trans_id  106 non-null    object 
 1   category  106 non-null    object 
 2   price     103 non-null    float64
 3   qty       106 non-null    int64  
dtypes: float64(1), int64(1), object(2)
memory usage: 3.4+ KB

DataFrame Description:


,price,qty
count,103.000000,106.000000
mean,9819.646699,2.660377
std,98521.926217,1.399910
min,-50.000000,1.000000
25%,74.305000,1.000000
50%,100.200000,3.000000
75%,138.510000,4.000000
max,1000000.000000,10.000000



Missing Values:


,0
trans_id,0
category,0
price,3
qty,0


**Step 3: The Cleaning Pipeline**

In [17]:
df_cleaned = df.copy() # Initialize df_cleaned from df
# Address negative price the -50.0 and extreme price like 1,000,000
# Remove prices less than or equal to 0, and excessively high prices
initial_rows_after_imputation = len(df_cleaned)
df_cleaned = df_cleaned[(df_cleaned['price'] > 0) & (df_cleaned['price'] < 5000)]

print(f"Removed {initial_rows_after_imputation - len(df_cleaned)} records due to price outliers.")
print("DataFrame after outlier mitigation (first 5 rows):")
display(df_cleaned.head())

Removed 7 records due to price outliers.
DataFrame after outlier mitigation (first 5 rows):


,trans_id,category,price,qty
0,T-101,Electronics,500.00,1
1,T-102,ELEC,450.00,2
3,T-104,Home & Kitchen,25.00,10
4,T-101,Electronics,500.00,1
6,T-106,Books,129.11,2


**Step 4: Validation & Reflection**

### Action 4.1: Check for positive prices and quantities

In [18]:
print("Checking for non-positive prices in cleaned data:")
price_check = df_cleaned[df_cleaned['price'] <= 0]
if price_check.empty:
    print("All prices in df_cleaned are positive.")
else:
    print("Found non-positive prices:")
    display(price_check)

print("\nChecking for non-positive quantities in cleaned data:")
qty_check = df_cleaned[df_cleaned['qty'] <= 0]
if qty_check.empty:
    print("All quantities in df_cleaned are positive.")
else:
    print("Found non-positive quantities:")
    display(qty_check)

Checking for non-positive prices in cleaned data:
All prices in df_cleaned are positive.

Checking for non-positive quantities in cleaned data:
All quantities in df_cleaned are positive.


### Action 4.2: Compare total revenue before and after cleaning

In [19]:
df['revenue'] = df['price'] * df['qty']
df_cleaned['revenue'] = df_cleaned['price'] * df_cleaned['qty']

total_revenue_before = df['revenue'].sum()
total_revenue_after = df_cleaned['revenue'].sum()

print(f"Total Revenue Before Cleaning: ${total_revenue_before:,.2f}")
print(f"Total Revenue After Cleaning: ${total_revenue_after:,.2f}")
print(f"Difference in Total Revenue: ${total_revenue_before - total_revenue_after:,.2f}")

Total Revenue Before Cleaning: $2,028,554.46
Total Revenue After Cleaning: $28,737.74
Difference in Total Revenue: $1,999,816.72
